# Forget-MI LoKU — Machine Unlearning Pipeline

Notebook này thực hiện toàn bộ pipeline:
1. Kết nối Drive & Clone code
2. Giải nén dữ liệu & models
3. Tiền xử lý (tạo all_data.tsv)
4. Huấn luyện LoKU Unlearning

In [ ]:
# ====================================
# CELL 1: Kết nối Drive & Clone Code
# ====================================
from google.colab import drive
import os

# 1. Mount Google Drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive', force_remount=True)
else:
    print("✅ Google Drive đã được kết nối!")

# 2. Clone Code từ GitHub
%cd /content
!rm -rf Forget-MI-LoKU
!git clone https://github.com/nhnhu146/Forget-MI-LoKU.git
%cd Forget-MI-LoKU

# 3. Cài đặt thư viện
!pip install -q pydicom scikit-image wandb pyyaml pandas
!pip install -q "transformers==4.38.0" "peft==0.10.0" "accelerate==0.27.0"

print("\n✅ Môi trường và mã nguồn đã sẵn sàng!")

In [ ]:
# ====================================
# CELL 2: Giải nén Data & Models
# ====================================
!python setup_data.py

In [ ]:
# ====================================
# CELL 3: Tiền xử lý & Thiết lập Output
# ====================================
import os
import shutil

# Tạo all_data.tsv từ các file báo cáo
!python make_tsv.py

# Xóa cache features cũ (nếu có)
!rm -f ./data/metadata/cachedfeatures_train_seqlen-*
!rm -f ./data/metadata/cachednoisyfeatures_train_seqlen-*

# Kết nối thư mục Output với Drive để lưu bền vững
DRIVE_RESULTS = "/content/drive/MyDrive/Forget-MI-Project/unlearning_output"
os.makedirs(DRIVE_RESULTS, exist_ok=True)

if os.path.exists("unlearning_output"):
    if os.path.islink("unlearning_output"): os.unlink("unlearning_output")
    else: shutil.rmtree("unlearning_output")

!ln -s "{DRIVE_RESULTS}" ./unlearning_output
print(f"\n✅ Output sẽ được lưu tại: {DRIVE_RESULTS}")

In [ ]:
# ====================================
# CELL 4: Chạy LoKU Unlearning
# ====================================
!PYTHONPATH=. WANDB_MODE=disabled python training/forgetmi_loku.py --config config.yaml